# Evaluation Observation — AMLK Hebrew Summarization

**Stage:** `evaluation-observation`. Where this sits in the project: *after* the training job has
produced test-set predictions (`predictions-{finetuned,base,gemini}.jsonl` on the model repo) and
*before* the D1 report tables. The D1 tables (`outputs/results/d1-tables.md`) show only aggregate
numbers; this notebook shows the **process** — for each example: the article, the model's summary,
the reference, what the LLM judge said (faithfulness / fluency), and the error-analysis failure
labels — by calling the **real** evaluation functions live (`evaluation/evaluate.py`,
`evaluation/error_analysis.py`, `evaluation/infer.py`). Nothing here re-implements scoring logic.

**Execution environment:** Google Colab. The live-generation cell (the last code cell) needs a
**T4 GPU** (it loads Qwen3-2B + the LoRA adapter); the judge / error-analysis / browse cells are
API + CPU only.

An agent can run this notebook cell-by-cell on a persistent Colab session with
`python -m scripts.run_nb_cell <this.ipynb> --session <name> --range A:B` (see
`scripts/run_nb_cell.py` and the `colab-cli` skill).

In [ ]:
# [0] Bootstrap: clone the repo, install deps, expose the package, load secrets.
import os, sys, subprocess

REPO_URL = "https://github.com/avrymi-asraf/AMLK-the-true-title.git"
REPO_DIR = "AMLK-the-true-title"
if os.path.basename(os.getcwd()) != REPO_DIR:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt",
                "sentencepiece"], check=True)
# Colab preinstalls torchao 0.10.0; the pip'd peft (>=0.19) raises ImportError at import time
# because it wants torchao>=0.16. We quantize with bitsandbytes, not torchao, so just remove it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao"], check=False)
sys.path.insert(0, os.getcwd())

# Colab wraps `google.generativeai` (via _GenerativeAIImportHook) to proxy every call through its
# runtime service at localhost:45545; on an account without runtime-proxy permission that proxy
# hangs forever instead of erroring (the FutureWarning you see comes from the same hook). Drop the
# finder before the SDK is first imported so it talks to Google directly — verified to fix the hang
# (REST + de-hooked SDK both answered in ~0.5s). Harmless off Colab (no such finder exists there).
sys.meta_path = [f for f in sys.meta_path if type(f).__name__ != "_GenerativeAIImportHook"]
for _m in [m for m in sys.modules if m.startswith("google.generativeai")]:
    del sys.modules[_m]


def get_secret(key: str) -> str:
    """env var -> /content/.env (agent path) -> Colab userdata (human) -> prompt.

    Agent-safe order: never relies on userdata first, and the agent injects keys via an uploaded
    /content/.env so the literal key never appears in an `exec` payload or `colab log`.
    """
    if os.environ.get(key):
        return os.environ[key]
    env_file = "/content/.env"
    if os.path.exists(env_file):
        for line in open(env_file, encoding="utf-8"):
            if "=" in line and not line.lstrip().startswith("#"):
                k, v = line.split("=", 1)
                if k.strip() == key:
                    return v.strip().strip("'\"")
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val:
            return val
    except Exception:
        pass
    return input(f"Enter {key}: ")


for _k in ("HF_TOKEN", "GEMINI_API_KEY"):
    os.environ[_k] = get_secret(_k)
print("secrets present:", {k: bool(os.environ.get(k)) for k in ("HF_TOKEN", "GEMINI_API_KEY")})

In [2]:
# [1] Load the test split and the existing model predictions from the Hub.
import json
from datasets import load_from_disk
from huggingface_hub import hf_hub_download, snapshot_download

from training.config import model_repo, dataset_repo
from evaluation.predict import strip_think

HF_USER = "avreymi"
VARIANT = "whole"
SYSTEMS = ["finetuned", "base", "gemini"]
MODEL_REPO = model_repo(HF_USER, VARIANT)
DATASET_REPO = dataset_repo(HF_USER, VARIANT)
token = os.environ["HF_TOKEN"]

# Where each system's predictions live on the model repo: the training job pushes finetuned/base
# to the root; the eval job pushes the Gemini baseline under reports/ (see eval_hf_job.push()).
PRED_PATHS = {
    "finetuned": "predictions-finetuned.jsonl",
    "base": "predictions-base.jsonl",
    "gemini": "reports/predictions-gemini.jsonl",
}

# Test split — the same Arrow splits training used.
data_dir = snapshot_download(DATASET_REPO, repo_type="dataset", token=token)
test_ds = load_from_disk(os.path.join(data_dir, "test"))


def load_predictions(system: str) -> list[dict]:
    """The model's output on the test set, produced by the training/eval jobs. strip_think drops
    any leaked Qwen3 <think> reasoning so we see the summary, exactly as the metrics do."""
    path = hf_hub_download(MODEL_REPO, PRED_PATHS[system], repo_type="model", token=token)
    rows = [json.loads(line) for line in open(path, encoding="utf-8")]
    for r in rows:
        r["prediction"] = strip_think(r["prediction"])
    return rows


preds = {s: load_predictions(s) for s in SYSTEMS}
print({s: len(preds[s]) for s in SYSTEMS}, "| test split:", len(test_ds))

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

predictions-finetuned.jsonl:   0%|          | 0.00/7.10M [00:00<?, ?B/s]

predictions-base.jsonl:   0%|          | 0.00/7.15M [00:00<?, ?B/s]

predictions-gemini.jsonl:   0%|          | 0.00/7.85M [00:00<?, ?B/s]

{'finetuned': 1000, 'base': 1000, 'gemini': 1000} | test split: 1000


In [3]:
# [2] Display helpers — make the per-example process readable.
def _block(title: str, text: str, width: int = 100, clip: int = 1200) -> None:
    body = text if len(text) <= clip else text[:clip] + " …"
    print(f"\n── {title} " + "─" * max(0, width - len(title) - 4))
    print(body)


def show_example(row: dict, judge: dict | None = None, labels: list | None = None,
                 article_clip: int = 1500) -> None:
    """Article -> model summary -> reference, plus the judge verdict and failure labels if given."""
    _block("ARTICLE", row["text"], clip=article_clip)
    _block(f"MODEL SUMMARY [{row.get('model', '?')}]", row["prediction"])
    _block("REFERENCE", row["reference"])
    if judge is not None:
        print(f"\n  JUDGE: faithfulness={judge.get('faithfulness')}  fluency={judge.get('fluency')}")
    if labels is not None:
        print(f"  FAILURE LABELS: {labels or '(none)'}")
    print("=" * 100)


def show_comparison(idx: int) -> None:
    """Same article, every system's summary side by side."""
    _block("ARTICLE", preds[SYSTEMS[0]][idx]["text"], clip=1500)
    _block("REFERENCE", preds[SYSTEMS[0]][idx]["reference"])
    for s in SYSTEMS:
        _block(f"SUMMARY [{s}]", preds[s][idx]["prediction"])
    print("=" * 100)

In [7]:
# [3] Browse a few examples across all three systems (instant — uses the loaded predictions).
import random

random.seed(0)
n = min(len(preds[s]) for s in SYSTEMS)
for idx in random.sample(range(n), 3):
    print(f"\n########## example #{idx} ##########")
    show_comparison(idx)


########## example #864 ##########

── ARTICLE ─────────────────────────────────────────────────────────────────────────────────────────
יותר מחודשיים לאחר שאיל ינון היועץ המשפטי של הכנסת סיים את תפקידו, הכנסת עדיין לא מקדמת מינוי יועץ חדש. בדיקת "שקוף" מגלה כי יו"ר הכנסת, ח"כ יריב לוין, אינו ממהר למנות יועץ משפטי. בינתיים מכהנת  בתפקיד ממלאת המקום, עו"ד שגית אפיק, אותה מינה יו"ר הכנסת הקודם יולי אדלשטיין ימים ספורים לפני שהתפטר. עד לפני חודש, התהליך התעכב לאור ההמתנה לקביעת ועדות הכנסת החדשות. כעת הכנסת חזרה לעבוד, הוועדות נקבעו, אך נראה שאיש לא ממהר למנות יועץ משפטי קבוע. מתברר שאפילו לא נערכה פנייה לנשיאת בית המשפט העליון אסתר חיות, כדי למנות יו"ר לוועדה שתפקידה למיין את המועמדים. העיכוב במינוי לא היה הפתעה: בינואר האחרון חשפנו כי הכנסת לא תצליח למצוא מחליף ליועץ המשפטי בזמן. אך מה שהחל בפער זמנים צפוי מראש, ממשיך כעת בגרירת רגליים ללא סיבה נראית לעין. תפקיד היועץ המשפטי לכנסת הוא תפקיד מפתח במשכן הכנסת בפרט ובמנגנון הדמוקרטי בכלל. הוא יכול למנוע תהליכי חקיקה חפוזים ואנטי דמוקרטיים

In [5]:
# [4] LIVE LLM judge on a small sample, per system — the real evaluate.judge_with_llm().
from evaluation.evaluate import judge_with_llm

SAMPLE_N = 5
sample = {s: preds[s][:SAMPLE_N] for s in SYSTEMS}
judge = {s: judge_with_llm(sample[s]) for s in SYSTEMS}

for s in SYSTEMS:
    print(f"\n########## judge — {s} ##########")
    print("means:", judge[s]["faithfulness_mean"], "faith /", judge[s]["fluency_mean"], "flu")
    for row, sc in zip(sample[s], judge[s]["per_example"]):
        show_example(row, judge=sc)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


KeyboardInterrupt: 

In [ ]:
# [5] LIVE error analysis on the same sample — the real error_analysis functions.
import google.generativeai as genai
from evaluation.predict import GEMINI_MODEL
from evaluation.error_analysis import label_predictions, failure_rates

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
gemini = genai.GenerativeModel(GEMINI_MODEL)

for s in SYSTEMS:
    labelled = label_predictions(sample[s], gemini)
    print(f"\n########## error analysis — {s} ##########")
    print("failure rates:", failure_rates(labelled))
    for row in labelled:
        show_example(row, labels=row["labels"])

In [ ]:
# [6] LIVE generation on GPU — the fine-tuned model AND the zero-shot base (adapter off),
# then judge + error-label the fresh fine-tuned output with the same real functions.
import google.generativeai as genai
from evaluation.infer import load_finetuned_model, generate_summaries
from evaluation.evaluate import judge_with_llm
from evaluation.error_analysis import label_predictions
from evaluation.predict import GEMINI_MODEL, strip_think

model, tokenizer, device = load_finetuned_model(MODEL_REPO)
gen_ds = test_ds.select(range(min(4, len(test_ds))))
fresh = generate_summaries(model, tokenizer, gen_ds, VARIANT, device, label="finetuned")
with model.disable_adapter():
    fresh_base = generate_summaries(model, tokenizer, gen_ds, VARIANT, device, label="base")
for r in fresh + fresh_base:
    r["prediction"] = strip_think(r["prediction"])

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
_gemini = genai.GenerativeModel(GEMINI_MODEL)
fresh_judge = judge_with_llm(fresh)["per_example"]
fresh_labelled = label_predictions(fresh, _gemini)
for ft, bs, sc, lab in zip(fresh, fresh_base, fresh_judge, fresh_labelled):
    show_example(ft, judge=sc, labels=lab["labels"])
    _block("SUMMARY [base zero-shot]", bs["prediction"])

In [ ]:
# [7] Connect the numbers to the text: ROUGE + BERTScore on the freshly generated rows.
from evaluation.evaluate import compute_rouge, compute_bertscore

print("finetuned ROUGE:", compute_rouge(fresh))
print("finetuned BERTScore (CPU):", compute_bertscore(fresh))